In [3]:
import pandas as pd

df = pd.read_csv("results/results.csv")
print(df.shape)
df[["config_id", "Language", "Property", "NumberType", "Distribution",
    "Repetitions", "Flags", "machine", "run", "percentage"]].head()

(630, 25)


,config_id,Language,Property,NumberType,Distribution,Repetitions,Flags,machine,run,percentage
0,llm_c,C,AssocAdd,Float64,Wide,R1000000,O0,aarch64,0,93.0600
1,llm_c,C,AssocAdd,Float64,Wide,R1000000,O0,aarch64,1,93.0885
2,llm_c,C,AssocAdd,Float64,Wide,R1000000,O0,aarch64,2,93.0454
3,llm_c,C,AssocAdd,Float64,Wide,R1000000,O0,aarch64,3,93.0799
4,llm_c,C,AssocAdd,Float64,Wide,R1000000,O0,aarch64,4,93.0545


In [4]:
df.groupby(["machine", "Language"]).size()

machine  Language
aarch64  C           175
         Python      140
x86_64   C           175
         Python      140
dtype: int64

In [5]:
for col in ["Property", "NumberType", "Distribution", "Repetitions",
            "Language", "Flags", "machine"]:
    print(df.groupby(col)["percentage"].agg(["mean", "std", "count"]).round(2), "\n")

                 mean    std  count
Property                           
AssocAdd        85.58  10.56    200
AssocMul        67.92   9.70    240
Distributivity  68.37  15.01    190 

             mean    std  count
NumberType                     
Float32     67.35   6.20    120
Float64     75.15  15.25    510 

               mean    std  count
Distribution                     
Unit          75.45  11.54    320
Wide          71.82  16.52    310 

              mean    std  count
Repetitions                     
R10000       73.90  13.56    310
R1000000     73.43  15.03    320 

           mean    std  count
Language                     
C         78.13  16.18    350
Python    68.08   8.85    280 

                 mean    std  count
Flags                              
O0              71.74  11.16     70
O2              66.11   7.84     90
O2FastMath     100.00   0.00    100
O2MarchNative   70.80   9.93     90 

          mean    std  count
machine                     
aarch64  73.52  14

In [5]:
ok = df[df["Flags"] != "O2FastMath"]
for col in ["Property", "NumberType", "Distribution", "Repetitions", "Language", "machine"]:
    print(ok.groupby(col)["percentage"].agg(["mean", "std", "count"]).round(2), "\n")

                 mean   std  count
Property                          
AssocAdd        80.77  7.48    150
AssocMul        65.01  0.37    220
Distributivity  62.44  6.62    160 

             mean    std  count
NumberType                     
Float32     67.35   6.20    120
Float64     69.09  10.09    410 

               mean    std  count
Distribution                     
Unit          71.94   7.32    280
Wide          65.05  10.07    250 

              mean    std  count
Repetitions                     
R10000       68.89   7.91    260
R1000000     68.51  10.60    270 

           mean   std  count
Language                    
C         69.38  9.89    250
Python    68.08  8.85    280 

          mean   std  count
machine                    
aarch64  68.52  9.10    265
x86_64   68.86  9.65    265 



In [6]:
g = df.groupby(["config_id", "machine", "Repetitions"])["percentage"].std().reset_index()
g.groupby("Repetitions")["percentage"].mean().round(3)

Repetitions
R10000      0.377
R1000000    0.036
Name: percentage, dtype: float64

In [7]:
c = df[(df["Language"] == "C") & (df["Property"] == "AssocAdd")]
c.groupby(["Distribution", "Flags", "machine"])["percentage"].mean().round(2).unstack("machine")

machine                     aarch64  x86_64
Distribution Flags                         
Unit         O0               82.70   82.70
             O2FastMath      100.00  100.00
             O2MarchNative    82.90   82.90
Wide         O0               93.07   93.07
             O2               67.32   93.06
             O2FastMath      100.00  100.00

## Test causal : la contraction FMA

Hypothèse : l'écart entre ARM et x86 (93,06 % contre 67,32 % avec `-O2` sur la plage Wide) vient de la fusion `r*(high-low)+low` en un seul arrondi (FMA), que gcc fait par défaut sur ARM dès `-O2`, mais sur x86 seulement avec `-march=native`.

Expérience (workflow `causal`, programme `exp/assoc_llm.c`, graine 42, 1 000 000 tirages, plage Wide) : compilation sur les deux architectures avec et sans `-ffp-contract=off`.

In [8]:
pd.read_csv("results/causal.csv")

,machine,gcc,flags,percentage
0,aarch64,13,-O2 -ffp-contract=off,93.0794
1,aarch64,13,-O2 -march=native,67.3535
2,aarch64,13,-O2,67.3535
3,x86_64,13,-O2 -ffp-contract=off,93.0794
4,x86_64,13,-O2 -march=native,67.3535
5,x86_64,13,-O2,93.0794


Sans fusion (`-ffp-contract=off`) : 93,0794 % sur les deux architectures. Avec fusion (`-O2` sur ARM, `-O2 -march=native` sur x86) : 67,3535 % sur les deux. La cause est donc la fusion, pas l'architecture elle-même. Limite : testé sur une seule propriété (associativité de +) et une seule version de gcc (13).

In [10]:
from sklearn.tree import DecisionTreeRegressor, export_text

factors = ["Property", "NumberType", "Distribution", "Repetitions", "Seed",
           "Language", "PythonVersion", "GccVersion", "Flags", "machine"]
X = pd.get_dummies(df[factors].fillna("none")).astype(int)
tree = DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=0)
tree.fit(X, df["percentage"])
print("R2:", round(tree.score(X, df["percentage"]), 3))
print(export_text(tree, feature_names=list(X.columns)))

R2: 0.983
|--- Flags_O2FastMath <= 0.50
|   |--- Property_AssocAdd <= 0.50
|   |   |--- Distribution_Wide <= 0.50
|   |   |   |--- Property_Distributivity <= 0.50
|   |   |   |   |--- Repetitions_R10000 <= 0.50
|   |   |   |   |   |--- value: [65.17]
|   |   |   |   |--- Repetitions_R10000 >  0.50
|   |   |   |   |   |--- value: [64.97]
|   |   |   |--- Property_Distributivity >  0.50
|   |   |   |   |--- Flags_O2MarchNative <= 0.50
|   |   |   |   |   |--- value: [68.88]
|   |   |   |   |--- Flags_O2MarchNative >  0.50
|   |   |   |   |   |--- value: [76.49]
|   |   |--- Distribution_Wide >  0.50
|   |   |   |--- Property_Distributivity <= 0.50
|   |   |   |   |--- NumberType_Float64 <= 0.50
|   |   |   |   |   |--- value: [65.08]
|   |   |   |   |--- NumberType_Float64 >  0.50
|   |   |   |   |   |--- value: [64.89]
|   |   |   |--- Property_Distributivity >  0.50
|   |   |   |   |--- NumberType_Float32 <= 0.50
|   |   |   |   |   |--- value: [58.38]
|   |   |   |   |--- NumberType_F

In [11]:
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(8).round(3)

Flags_O2FastMath           0.651
Property_AssocAdd          0.241
NumberType_Float32         0.037
Property_Distributivity    0.029
Distribution_Wide          0.027
Seed_Seed42                0.009
Flags_O2MarchNative        0.004
Distribution_Unit          0.003
dtype: float64

In [12]:
sub = df[(df["Property"] == "AssocAdd") & (df["NumberType"] == "Float64")
         & (df["Distribution"] == "Wide") & (df["Flags"] != "O2FastMath")]
sub.groupby(["Seed", "Language", "Flags", "machine"], dropna=False)["percentage"].agg(["mean", "count"]).round(2)

mean  count
Seed          Language Flags machine              
Seed123456789 C        O0    aarch64  93.07      5
                             x86_64   93.07      5
              Python   NaN   aarch64  93.09      5
                             x86_64   93.09      5
Seed42        C        O2    aarch64  67.32      5
                             x86_64   93.06      5

## Lecture de l'arbre de décision

Arbre de régression de profondeur 5 sur les 630 lignes (R² = 0,983).

- `-ffast-math` est le facteur le plus important (importance 0,65) : toutes ces lignes donnent 100 %, car le compilateur réorganise les additions et l'égalité devient toujours vraie. La propriété n'est alors plus testée.
- La propriété testée vient ensuite (0,24 pour l'associativité de +). L'associativité de × reste autour de 65 % quels que soient les autres réglages.
- Float32 baisse l'associativité de + (74,5 % contre 82,8 % sur la plage Unit) et la distributivité (52,7 % contre 58,4 % sur Wide).
- Le nombre de tirages, le langage, les versions de Python et de gcc et la machine n'ont pas d'effet sur la moyenne dans l'arbre. Le nombre de tirages ne change que la dispersion : 0,377 point d'écart-type entre lancements à 10 000 tirages, 0,036 à 1 000 000.
- Le découpage de l'arbre sur la graine est un artefact : dans la tranche associativité de + / Float64 / Wide, la graine 42 donne 93,06 % sur x86, comme la graine 123456789 (93,07 %). Seule la ligne ARM avec -O2 donne 67,32 %, et elle a la graine 42 dans cet échantillon.
- Limite : l'arbre a au plus 5 niveaux et il ne montre ni la machine ni les options de compilation pour l'effet FMA, qui ne touche que 5 lignes dans la tranche étudiée. Un R² élevé ne garantit donc pas que tout est expliqué : c'est le test causal qui a révélé cet effet.

## Recommandations pour obtenir un résultat stable

1. Ne jamais compiler avec `-ffast-math` pour tester une propriété arithmétique : le résultat est toujours 100 %.
2. Contrôler la contraction FMA : compiler avec `-ffp-contract=off` donne 93,0794 % sur ARM comme sur x86 (test causal), alors que sans cette option le résultat dépend de l'architecture et de `-march=native` (67,35 % ou 93,08 %).
3. Utiliser au moins 1 000 000 de tirages : l'écart-type entre lancements passe de 0,38 à 0,04 point.
4. Fixer et publier la graine, la plage des valeurs et la façon de fabriquer les nombres aléatoires (distribution, opérations utilisées), car ces choix changent la réponse.
5. Les versions de Python (3.10, 3.14) et de gcc (12, 15) n'ont pas changé les résultats : il n'est pas indispensable de les figer pour cette expérience, même si le faire reste utile pour la reproductibilité.

In [13]:
import random
from fractions import Fraction

def pct(generate, n=100_000, seed=42):
    rng = random.Random(seed)
    eq = 0
    for _ in range(n):
        x, y, z = generate(rng), generate(rng), generate(rng)
        eq += (x + y) + z == x + (y + z)
    return 100 * eq / n

lo, hi = -1e6, 1e6
unfused = lambda rng: rng.random() * (hi - lo) + lo
fused = lambda rng: float(Fraction(rng.random()) * Fraction(hi - lo) + Fraction(lo))
print("deux arrondis:", pct(unfused), "| un seul arrondi (FMA émulé):", pct(fused))

deux arrondis: 92.979 | un seul arrondi (FMA émulé): 67.224


## Explication des résultats obtenus aux TP 1 et 2

- TP 1 (83,01 %) : Python, valeurs dans [0, 1), 10 000 tirages, graine 42. Avec 1 000 000 de tirages la même configuration donne environ 82,8 %, donc l'écart de 0,2 point est du bruit statistique (environ ±0,4 point à 10 000 tirages).
- TP 2, LLM en C et LLM en Python (93,06 %) : valeurs dans [-10^6, 10^6], 1 000 000 de tirages. Les deux programmes donnent le même pourcentage malgré des générateurs différents (xorshift64, Mersenne Twister), ce qui montre que le langage et le générateur ne comptent pas ici.
- L'écart entre 83 % et 93 % ne vient pas d'une propriété de l'addition elle-même : le même programme C compilé avec fusion FMA donne environ 67 % au lieu de 93 %. La façon de fabriquer les nombres (`r*(high-low)+low`, avec deux arrondis ou un seul) change la réponse. Explication probable, non testée séparément : avec deux arrondis, les nombres générés sur la plage large ont des bits de poids faible nuls, donc les additions sont plus souvent exactes. La simulation ci-dessus va dans ce sens (92,98 % contre 67,22 %).
- Sur la plage [0, 1), la fusion n'a aucun effet (r*1+0 est exact), ce qui explique que les deux architectures donnent les mêmes valeurs.
- Limites : échantillon de 60 configurations (peu de lignes touchées par l'effet FMA), test causal fait avec une seule version de gcc (13) et une seule propriété.